# Week 62, Day 1: Names and references

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w062d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
a = [1, 2]
b = a
b += [3]
print(a)

x = (1, 2)
y = x
y += (3,)
print(x, y)

def grow(items):
    items.append(4)
    items = [0]
    return items

print(grow(a), a)

In [ ]:
import math

count = 3
def helper():
    pass

ns = globals()
print("count" in ns, ns["count"], ns["helper"] is helper, ns["math"] is math)

class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

p = Point(1, 2)
print(vars(p), p.__dict__ is vars(p))
vars(p)["z"] = 3                       # writing to the namespace is setting an attribute
print(p.z)

items = [1, 2, 3]
alias = items
del items                              # removes the name, not the list
print(alias)

In [ ]:
a = b = []
a.append("shared")
print(b)

class Team:
    def __init__(self, members):
        self.members = members            # shared with the caller
        self.safe = list(members)         # a copy of the caller's list

names = ["ada", "ben"]
team = Team(names)
names.append("cy")
print(team.members, team.safe)

In [ ]:
def add_sale(sales, amount):
    sales.append(amount)                  # mutates the caller's list

def reset(sales):
    sales = []                            # only rebinds the local name

def reset_properly(sales):
    sales.clear()                         # mutates: empties the caller's list

day = [5, 3]
add_sale(day, 9)
reset(day)
print(day)
reset_properly(day)
print(day)

def tagged(tag, tags=None):
    tags = [] if tags is None else tags
    tags.append(tag)
    return tags

print(tagged("a"), tagged("b"))

In [ ]:
a = [1]
alias = a
a += [2]
print(alias, a is alias)
a = a + [3]
print(alias, a is alias)

s = "ab"
s_alias = s
s += "c"
print(s_alias, s)

t = ([1],)
try:
    t[0] += [2]
except TypeError as e:
    print("TypeError:", e)
print(t)

> **Key idea.** A name is an entry in a namespace, and namespaces are dicts: globals(), vars(obj), a class's __dict__. Assignment, for, def, class, import and as all bind names; del removes a name, not the object. Assignment never copies: every binding, argument, return value and container slot is a reference, and copies happen only when asked for. Functions receive references (call by sharing): mutating changes the caller's object, rebinding changes only the local name, and mutable defaults are shared between calls. x += y calls __iadd__ when it exists (mutating lists in place) and otherwise rebinds to x + y (new ints, strings, tuples). Draw the reference graph to see who shares what.

### Exercise: Who shares what?

Write `shared_names(namespace)`: given a dict from names to objects, return a sorted list of groups (each a sorted list of names) where every name in the group refers to the **same** object, keeping only groups of two or more. Then `bound_by(source)`: the sorted list of names a piece of code binds at module level, found by running it with `exec` in a fresh dict and listing the keys (ignore names starting with `__`).

In [ ]:
def shared_names(namespace):
    ___

def bound_by(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_namespace():\n    \"identity and namespaces\"\n    a = [1]\n    ns = {\"a\": a, \"b\": a, \"c\": [1], \"d\": \"x\", \"e\": a, \"f\": None, \"g\": None}\n    assert shared_names(ns) == [[\"a\", \"b\", \"e\"], [\"f\", \"g\"]]\n    assert shared_names({}) == []\n    code = \"import math\\nx = 1\\nfor i in range(2):\\n    pass\\ndef f():\\n    y = 2\\nclass C:\\n    pass\\nwith open('t.txt', 'w') as fh:\\n    pass\\n\"\n    assert bound_by(code) == [\"C\", \"f\", \"fh\", \"i\", \"math\", \"x\"]")

### Exercise: Mutate or return new

Write three functions on lists of numbers: `doubled(nums)` returns a **new** list (the argument unchanged); `double_in_place(nums)` changes the caller's list and returns `None`; and `replace_all(nums, value)` makes **every** element of the caller's list equal `value` without rebinding (so all aliases see it).

In [ ]:
def doubled(nums):
    ___

def double_in_place(nums):
    ___

def replace_all(nums, value):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_sharing():\n    \"call by sharing\"\n    a = [1, 2, 3]\n    alias = a\n    assert doubled(a) == [2, 4, 6] and a == [1, 2, 3]\n    assert double_in_place(a) is None and alias == [2, 4, 6] and a is alias\n    replace_all(a, 0)\n    assert alias == [0, 0, 0]\n    empty = []\n    double_in_place(empty)\n    assert empty == []")

### Exercise: In place or new

Write a class `Bag(items=())` holding a list of items. `bag + other` returns a **new** Bag with both bags' items (neither changed); `bag += other` adds the other bag's items **in place** (define `__iadd__`, returning `self`); `len`, `==` (same items in the same order) and `items()` (a copy of the list) work too. Then `Tally(n)`, an **immutable** counter with only `__add__` (adding a whole number gives a new Tally) and `value`: so `t += 1` rebinds.

In [ ]:
class Bag:
    def __init__(self, items=()):
        ___

class Tally:
    def __init__(self, n=0):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_iadd():\n    \"__iadd__ versus __add__\"\n    a = Bag([\"tea\"])\n    alias = a\n    b = a + Bag([\"jam\"])\n    assert b.items() == [\"tea\", \"jam\"] and a.items() == [\"tea\"] and b is not a\n    a += Bag([\"milk\"])\n    assert alias.items() == [\"tea\", \"milk\"] and a is alias and len(a) == 2\n    assert Bag([1]) == Bag([1]) and Bag([1]) != Bag([2])\n    a.items().append(\"hack\")\n    assert len(a) == 2\n    t = Tally(1)\n    original = t\n    t += 2\n    assert t.value == 3 and original.value == 1 and t is not original\n    assert not hasattr(Tally, \"__iadd__\")")

### Exercise: Rows that aren't shared

Write `make_grid(rows, cols, fill)`: a list of lists where every row is a **separate** list, and `shared_rows(grid)`: the number of rows that are the same object as some **earlier** row (so `[[0] * 3] * 3` gives 2). Then `fix_grid(grid)`: make every row separate **in place** (the outer list is the same object afterwards), keeping the values.

In [ ]:
def make_grid(rows, cols, fill):
    ___

def shared_rows(grid):
    ___

def fix_grid(grid):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_grid():\n    \"aliasing in nested lists\"\n    g = make_grid(3, 2, 0)\n    g[0][0] = 9\n    assert g == [[9, 0], [0, 0], [0, 0]] and shared_rows(g) == 0\n    bad = [[0] * 3] * 3\n    assert shared_rows(bad) == 2\n    outer = bad\n    fix_grid(bad)\n    assert bad is outer and shared_rows(bad) == 0\n    bad[1][1] = 5\n    assert bad == [[0, 0, 0], [0, 5, 0], [0, 0, 0]]\n    row = [1]\n    assert shared_rows([row, [1], row, row]) == 2")

### Exercise: A default that isn't shared

Write `add_score(name, score, board=None)`: add the score to the list for that name in the board (a dict from name to list), creating the board if none is given, and return the board. Calls without a board must **not** share one. Then `Basket(items=None)`: a class whose instances each get their **own** list, even when made without arguments, and which copies a list passed in so the caller's list isn't shared.

In [ ]:
def add_score(name, score, board=None):
    ___

class Basket:
    def __init__(self, items=None):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_defaults():\n    \"mutable defaults\"\n    b1 = add_score(\"ada\", 3)\n    b2 = add_score(\"ben\", 5)\n    assert b1 == {\"ada\": [3]} and b2 == {\"ben\": [5]}\n    shared = {}\n    add_score(\"cy\", 1, shared)\n    add_score(\"cy\", 2, shared)\n    assert shared == {\"cy\": [1, 2]}\n    x, y = Basket(), Basket()\n    x.items.append(\"tea\")\n    assert y.items == []\n    mine = [\"jam\"]\n    z = Basket(mine)\n    z.items.append(\"milk\")\n    assert mine == [\"jam\"] and z.items == [\"jam\", \"milk\"]")

### Bug hunt: One list, two names

`a = b = []` makes **one** list with two names, so adding a task to "today" also adds it to "tomorrow" and this prints `['call', 'shop'] ['call', 'shop']`. Make two separate lists so it prints `['call'] ['shop']`.

In [ ]:
today = tomorrow = []
today.append("call")
tomorrow.append("shop")
print(today, tomorrow)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"two lists\"\n    expect_output(\"['call'] ['shop']\")")

### Bug hunt: Rebinding the parameter

`clear_list` rebinds its parameter to a new empty list, which only moves the local name: the caller's list is untouched and this prints `[1, 2, 3]`. Empty the caller's list in place so it prints `[]`.

In [ ]:
def clear_list(items):
    items = []

data = [1, 2, 3]
clear_list(data)
print(data)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"mutate, don't rebind\"\n    expect_output(\"[]\")")

### Bug hunt: A copy that wasn't

The report stores the caller's list itself, so when the caller later adds to it, the "saved" report changes too: this prints `[10, 20, 30]`. Copy the list when saving so it prints `[10, 20]`.

In [ ]:
class Report:
    def __init__(self, figures):
        self.figures = figures

monthly = [10, 20]
saved = Report(monthly)
monthly.append(30)
print(saved.figures)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"copy on the way in\"\n    expect_output(\"[10, 20]\")")

### Puzzle: A safe default

Arrange the lines of a function that appends to a list, using a fresh list when none is given. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # a = [1]; b = a; b = b + [2]; a
q = ___   # a = [1]; b = a; b += [2]; a
r = ___   # s = "x"; t = s; t += "y"; s
s = ___   # True or False: del name deletes the object the name referred to
t = ___   # the exception from t = ([1],); t[0] += [2], as a string

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [1] and q == [1, 2] and r == \"x\", \"p, q and r\"\n    assert s is False and t == \"TypeError\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: drawing the reference graph

Given a namespace (a dict from names to objects), write tools that walk the objects reachable from it through lists, tuples, dicts (keys and values) and sets, treating everything else as a leaf:

- `reachable(namespace)`: the number of **distinct** objects reachable from the names (by identity), counting containers and leaves.
- `incoming(namespace)`: a dict from `id(obj)` to how many references point at that object, counting each name and each container slot (a dict entry counts its key and its value separately).
- `shared(namespace)`: the number of distinct objects with **more than one** incoming reference.
- `has_cycle(namespace)`: True if some container can reach itself by following references.

Walk without recursion (an explicit stack), and visit each object once even when it's shared or in a cycle.

In [ ]:
def reachable(namespace):
    ___

def incoming(namespace):
    ___

def shared(namespace):
    ___

def has_cycle(namespace):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_graph():\n    \"reference graphs\"\n    row = [0, 0]\n    grid = [row, row]\n    ns = {\"grid\": grid, \"row\": row, \"n\": 5}\n    counts = incoming(ns)\n    assert counts[id(row)] == 3 and counts[id(grid)] == 1\n    assert reachable(ns) == 4 and shared(ns) == 2\n    assert not has_cycle(ns)\n    loop = [1]\n    loop.append(loop)\n    assert has_cycle({\"loop\": loop}) and reachable({\"loop\": loop}) == 2 and incoming({\"loop\": loop})[id(loop)] == 2\n    a, b = {\"name\": \"a\"}, {\"name\": \"b\"}\n    a[\"friend\"], b[\"friend\"] = b, a\n    assert has_cycle({\"a\": a}) and not has_cycle({\"x\": [[1], [2]]})\n    deep = []\n    node = deep\n    for _ in range(5000):\n        nxt = []\n        node.append(nxt)\n        node = nxt\n    assert reachable({\"deep\": deep}) == 5001 and not has_cycle({\"deep\": deep})")

## Warm-up

Press X-ray on this one. `b += [3]` changed the list both names share; `y += (3,)` built a new tuple and moved only `y`; and inside `grow`, `append` changed the caller's list but `items = [0]` only moved the local name.

## Names live in namespaces

A **name** (a variable) is a label that refers to an **object**. Python keeps names in **namespaces**, and each namespace is, in fact, a dictionary from names to objects:

- a module's globals: `globals()`, also `module.__dict__`;
- a function call's locals (in a fast internal form, shown by `locals()`);
- an object's attributes: `obj.__dict__`, also `vars(obj)`;
- a class's attributes: `Cls.__dict__`.

Looking up `x` means finding `"x"` in the right namespace (Week 6's LEGB rule says which), and `obj.attr` looks in `vars(obj)` (and then the class). Many statements **bind** names, not just `=`: `for` binds its loop variable, `def` and `class` bind the new function or class, `import` binds the module, `with ... as` and `except ... as` bind their names. `del name` removes the **name** from the namespace; the object survives as long as anything else refers to it.

## Assignment never copies

`b = a` makes `b` refer to **the same object** as `a`: it never copies. That's true of every binding: passing an argument, returning a value, putting an object in a list (the list holds a reference), `for x in items` (each `x` refers to an item, not a copy of it). Copies happen only when you ask for one: `list(a)`, `a[:]`, `a.copy()`, `dict(d)`, `copy.copy`, `copy.deepcopy` (Thursday).

Consequences you've met:

- `a = b = []` makes **one** list with two names.
- `[[0] * 3] * 3` is three references to **one** row (Week 8).
- Storing a caller's list in an object (`self.items = items`) means the object and the caller share it; later changes through either show through both. Copy on the way in (`self.items = list(items)`) if that's not what you want.

## Calling a function

When you call `f(x)`, the parameter inside `f` becomes a new **name** for the **same object** the caller passed. Python's argument passing is sometimes called **call by sharing** (or "pass by object reference"): neither "by value" (no copy is made) nor "by reference" (the function can't rebind the caller's variable). So:

- **Mutating** the object (`items.append(...)`, `d[k] = v`, `obj.attr = v`) changes the caller's object: they're the same object.
- **Rebinding** the parameter (`items = [...]`) changes only the local name; the caller's name still refers to the old object.

That's also why mutable **default arguments** are shared between calls (Week 7): the default object is created once, when `def` runs, and every call that doesn't pass the argument gets a reference to that same object. Use `None` and create a fresh object inside.

## What += really does

`x += y` is **not** just shorthand for `x = x + y`. Python first tries `x.__iadd__(y)` (in-place add): mutable types like `list` define it to **change the object** and return it; then the name is bound to the result (the same object). Immutable types (`int`, `str`, `tuple`) have no `__iadd__`, so Python falls back to `x = x + y`, building a **new** object and rebinding the name.

So `a += [1]` changes the list every alias sees, while `a = a + [1]` builds a new list and changes only `a`. The same holds for `-=`, `*=`, `|=` and the rest (`__isub__`, `__imul__`, `__ior__`). And a famous puzzle: `t = ([1],); t[0] += [2]` **both** changes the inner list **and** raises `TypeError`, because the in-place add succeeds and then assigning the result back into the tuple fails.

## Reference graphs

To reason about sharing, draw the **reference graph**: names on the left, objects as boxes, an arrow for every reference (from a name, a list slot, a dict value, an attribute). An object can have many arrows coming in (it's shared), and arrows can form **cycles** (a list that contains itself, two objects that refer to each other). The X-ray view draws exactly this. Two questions the graph answers:

- **Who sees a change?** Everything with a path of arrows to the object that changed.
- **When can an object be freed?** When no path from any name reaches it any more (Wednesday's garbage collection).

`id(obj)` gives an object's identity (in CPython, its memory address), unique among objects alive at the same time; `a is b` means `id(a) == id(b)`. Use `id` to tell objects apart in a graph, never to store as a permanent label: once an object is freed, its id can be reused.

## Practice

## Quiz

## Daily challenge

## Before you go

Pick one of today's examples, press X-ray, and draw the reference graph on paper before looking at the X-ray view; then compare. Tomorrow: identity and mutability, and when `is` is the right test.